# Step 1: Import Libraries

In [1]:
import os
import pandas as pd
import numpy as np

from google.colab import drive

# Step 2: Load Image Records

Load the image records saved during data exploration, including image paths, class labels, image IDs, and patient IDs.

In [2]:
# Mount Google Drive
drive.mount("/content/drive")

# Set dataset path
csv_path = "/content/drive/MyDrive/COMP3710/project/dataset/processed/image_records.csv"

# Load CSV
df = pd.read_csv(
    csv_path,
    dtype={
        "image_id": str,
        "patient_id": str
    }
)

print("Total images:", len(df))
print("Missing patient IDs:", df["patient_id"].isna().sum())

print("\nFirst 5 records:")
print(df.head())

Mounted at /content/drive
Total images: 30521
Missing patient IDs: 0

First 5 records:
   split label        filename image_id  \
0  train    AD  218391_78.jpeg   218391   
1  train    AD  218391_79.jpeg   218391   
2  train    AD  218391_80.jpeg   218391   
3  train    AD  218391_81.jpeg   218391   
4  train    AD  218391_82.jpeg   218391   

                                                path  patient_id  
0  /content/drive/MyDrive/COMP3710/project/datase...  037_S_4001  
1  /content/drive/MyDrive/COMP3710/project/datase...  037_S_4001  
2  /content/drive/MyDrive/COMP3710/project/datase...  037_S_4001  
3  /content/drive/MyDrive/COMP3710/project/datase...  037_S_4001  
4  /content/drive/MyDrive/COMP3710/project/datase...  037_S_4001  


# Step 3: Check Original Patient Overlap

Identify unique patients in the original training and test sets and detect patient-level overlap.

In [3]:
# Get original train and test records
original_train = df[df["split"] == "train"]
original_test = df[df["split"] == "test"]

# Get unique patient IDs
train_subjects = set(original_train["patient_id"])
test_subjects = set(original_test["patient_id"])

# Find overlapping patients
overlap = train_subjects & test_subjects

print("Train patients:", len(train_subjects))
print("Test patients:", len(test_subjects))
print("Overlapping patients:", len(overlap))
print("Total unique patients:", len(train_subjects | test_subjects))

Train patients: 565
Test patients: 331
Overlapping patients: 216
Total unique patients: 680


# Step 4: Resolve Patient-Level Overlap

Assign all overlapping patients to the training candidate group.
Collect their images from both original folders.
Keep only original test-exclusive patients in the test set.

In [17]:
# Verify patient IDs and labels
assert df["patient_id"].notna().all()
assert set(df["label"]) == {"AD", "NC"}

label_counts = df.groupby("patient_id")["label"].nunique()
assert (label_counts == 1).all(), "Conflicting patient labels found."

# Collect all images, preserving their original paths
candidate_df = df.copy().reset_index(drop=True)

# Remove the previously identified duplicate record
candidate_df = candidate_df[
    candidate_df["filename"] != "976382_96 (1).jpeg"
].copy()

# Check that all image files exist
missing = ~candidate_df["path"].map(os.path.isfile)
print("Missing image files:", missing.sum())
assert not missing.any(), "Resolve missing image paths before splitting."

assert candidate_df["path"].is_unique
candidate_df = candidate_df.reset_index(drop=True)

print("Candidate images:", len(candidate_df))
print("Candidate patients:", candidate_df["patient_id"].nunique())

Missing image files: 0
Candidate images: 30520
Candidate patients: 680


# Step 5: Verify Final Train/Test Splits

Verify that the final training and test sets contain no overlapping patients, no conflicting patient labels, and no missing patient IDs.

In [5]:
# # Check patient overlap
# final_overlap = (
#     set(train_df["patient_id"])
#     &
#     set(test_df["patient_id"])
# )

# # Check missing patient IDs
# missing_train = train_df["patient_id"].isna().sum()
# missing_test = test_df["patient_id"].isna().sum()

# # Check that both classes are present
# print("Final Train class distribution:")
# print(train_df["label"].value_counts())

# print("\nFinal Test class distribution:")
# print(test_df["label"].value_counts())

# print("\nTrain/Test overlapping patients:", len(final_overlap))
# print("Missing Train patient IDs:", missing_train)
# print("Missing Test patient IDs:", missing_test)

# # Final checks
# assert len(final_overlap) == 0
# assert missing_train == 0
# assert missing_test == 0

# assert set(train_df["label"]) == {"AD", "NC"}
# assert set(test_df["label"]) == {"AD", "NC"}

# assert len(train_df) > 0
# assert len(test_df) > 0

# print("\nPASS: Patient-independent Train/Test splits verified.")

Final Train class distribution:
label
NC    14601
AD    12480
Name: count, dtype: int64

Final Test class distribution:
label
AD    2380
NC    1060
Name: count, dtype: int64

Train/Test overlapping patients: 0
Missing Train patient IDs: 0
Missing Test patient IDs: 0

PASS: Patient-independent Train/Test splits verified.


In [6]:
# original_patients = set(df["patient_id"])

# final_patients = (
#     set(train_df["patient_id"])
#     |
#     set(test_df["patient_id"])
# )

# print("Original unique patients:", len(original_patients))
# print("Final unique patients:", len(final_patients))

# assert original_patients == final_patients

# print("PASS: All original patients are represented.")

Original unique patients: 680
Final unique patients: 680
PASS: All original patients are represented.


# Step 5.1: Remove Missing Image Records

Check whether all image files exist and remove invalid records before saving the final train and test splits.

In [7]:
# # Check image files and remove missing records

# print("Train images before:", len(train_df))
# print("Test images before:", len(test_df))

# missing_train = []
# missing_test = []

# # Check training images
# for path in train_df["path"]:
#     if not os.path.isfile(path):
#         missing_train.append(path)

# # Check test images
# for path in test_df["path"]:
#     if not os.path.isfile(path):
#         missing_test.append(path)

# print("\nMissing train images:", len(missing_train))
# print("Missing test images:", len(missing_test))

# for path in missing_train:
#     print("Missing train:", path)

# for path in missing_test:
#     print("Missing test:", path)

# # Remove invalid records
# train_df = train_df[
#     ~train_df["path"].isin(missing_train)
# ].reset_index(drop=True)

# test_df = test_df[
#     ~test_df["path"].isin(missing_test)
# ].reset_index(drop=True)

# print("\nTrain images after:", len(train_df))
# print("Test images after:", len(test_df))

Train images before: 27081
Test images before: 3440

Missing train images: 0
Missing test images: 0

Train images after: 27081
Test images after: 3440


In [8]:
# # Find records related to the duplicated filename
# matches = train_df[
#     train_df["filename"].str.contains("976382_96", regex=False)
# ]

# print(matches[["filename", "path"]].to_string(index=False))

          filename                                                                                       path
976382_96 (1).jpeg /content/drive/MyDrive/COMP3710/project/dataset/raw/ADNI/AD_NC/train/NC/976382_96 (1).jpeg
    976382_96.jpeg     /content/drive/MyDrive/COMP3710/project/dataset/raw/ADNI/AD_NC/train/NC/976382_96.jpeg


In [9]:
# # Remove the duplicate image record

# print("Train images before:", len(train_df))

# duplicate_name = "976382_96 (1).jpeg"

# train_df = train_df[
#     train_df["filename"] != duplicate_name
# ].reset_index(drop=True)

# print("Train images after:", len(train_df))

# # Verify the original image is still included
# matches = train_df[
#     train_df["filename"].str.contains("976382_96", regex=False)
# ]

# print(matches[["filename", "path"]].to_string(index=False))

Train images before: 27081
Train images after: 27080
      filename                                                                                   path
976382_96.jpeg /content/drive/MyDrive/COMP3710/project/dataset/raw/ADNI/AD_NC/train/NC/976382_96.jpeg


#==============


In [16]:
# # Recover all training candidate patients
# candidate_patients = pd.concat(
#     [
#         train_df[["patient_id", "label"]],
#         val_df[["patient_id", "label"]]
#     ],
#     ignore_index=True
# ).drop_duplicates()

# print("Training candidate patients:", len(candidate_patients))
# print(candidate_patients["label"].value_counts())

# print("\nValidation patients by class:")
# print(
#     val_df[["patient_id", "label"]]
#     .drop_duplicates()["label"]
#     .value_counts()
# )

Training candidate patients: 565
label
NC    411
AD    154
Name: count, dtype: int64

Validation patients by class:
label
NC    82
AD    31
Name: count, dtype: int64


# Step 5.2: Create Patient-Independent Validation Set

Select 10% of training candidate patients for validation using
class-stratified random sampling. Keep all images from each patient
in the same split. The test set remains unchanged.

In [10]:
# from sklearn.model_selection import train_test_split

# # One label per patient
# assert (
#     train_df.groupby("patient_id")["label"].nunique() == 1
# ).all(), "Conflicting patient labels found."

# patient_df = train_df[
#     ["patient_id", "label"]
# ].drop_duplicates()

# # Randomly select 10% of patients for validation
# train_patients, val_patients = train_test_split(
#     patient_df["patient_id"],
#     test_size=0.20,
#     random_state=42,
#     stratify=patient_df["label"]
# )

# # Keep all images from each patient in the same split
# val_df = train_df[
#     train_df["patient_id"].isin(val_patients)
# ].copy()

# train_df = train_df[
#     train_df["patient_id"].isin(train_patients)
# ].copy()

# train_df = train_df.reset_index(drop=True)
# val_df = val_df.reset_index(drop=True)

# train_df["split"] = "train"
# val_df["split"] = "val"

# for name, records in [
#     ("Training", train_df),
#     ("Validation", val_df),
#     ("Test", test_df)
# ]:
#     print(f"\n{name} images:", len(records))
#     print(f"{name} patients:", records["patient_id"].nunique())
#     print(records["label"].value_counts())


Training images: 21980
Training patients: 452
label
NC    11780
AD    10200
Name: count, dtype: int64

Validation images: 5100
Validation patients: 113
label
NC    2820
AD    2280
Name: count, dtype: int64

Test images: 3440
Test patients: 115
label
AD    2380
NC    1060
Name: count, dtype: int64


# Step 5.3: Verify Patient Independence

In [11]:
# # Get patient IDs
# train_ids = set(train_df["patient_id"])
# val_ids = set(val_df["patient_id"])
# test_ids = set(test_df["patient_id"])

# print("Train / Validation overlap:", len(train_ids & val_ids))
# print("Train / Test overlap:", len(train_ids & test_ids))
# print("Validation / Test overlap:", len(val_ids & test_ids))

# # All three splits must contain different patients
# assert not (train_ids & val_ids)
# assert not (train_ids & test_ids)
# assert not (val_ids & test_ids)

# # Check image path uniqueness and separation
# train_paths = set(train_df["path"])
# val_paths = set(val_df["path"])
# test_paths = set(test_df["path"])

# assert train_df["path"].is_unique
# assert val_df["path"].is_unique
# assert test_df["path"].is_unique

# assert not (train_paths & val_paths)
# assert not (train_paths & test_paths)
# assert not (val_paths & test_paths)

# print("\nPASS: All datasets are patient-independent.")

Train / Validation overlap: 0
Train / Test overlap: 0
Validation / Test overlap: 0

PASS: All datasets are patient-independent.


In [18]:
from sklearn.model_selection import train_test_split

# One record per patient; sort for reproducibility
patient_df = (
    candidate_df[["patient_id", "label"]]
    .drop_duplicates()
    .sort_values("patient_id")
    .reset_index(drop=True)
)

# 70% training, 30% temporary holdout
train_patient_df, holdout_patient_df = train_test_split(
    patient_df,
    test_size=0.30,
    random_state=42,
    stratify=patient_df["label"]
)

# Divide the holdout equally: 15% validation, 15% test
val_patient_df, test_patient_df = train_test_split(
    holdout_patient_df,
    test_size=0.50,
    random_state=42,
    stratify=holdout_patient_df["label"]
)

# Keep every patient's images in exactly one split
def collect_images(patients, split_name):
    records = candidate_df[
        candidate_df["patient_id"].isin(patients["patient_id"])
    ].copy()

    records["split"] = split_name
    return records.reset_index(drop=True)

train_df = collect_images(train_patient_df, "train")
val_df = collect_images(val_patient_df, "val")
test_df = collect_images(test_patient_df, "test")

for name, records in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    patients = records[["patient_id", "label"]].drop_duplicates()

    print(f"\n{name} images:", len(records))
    print(f"{name} patients:", len(patients))
    print("Patient class distribution:")
    print(patients["label"].value_counts())
    print("Image class distribution:")
    print(records["label"].value_counts())


Train images: 21620
Train patients: 476
Patient class distribution:
label
NC    321
AD    155
Name: count, dtype: int64
Image class distribution:
label
NC    11000
AD    10620
Name: count, dtype: int64

Validation images: 4460
Validation patients: 102
Patient class distribution:
label
NC    69
AD    33
Name: count, dtype: int64
Image class distribution:
label
AD    2240
NC    2220
Name: count, dtype: int64

Test images: 4440
Test patients: 102
Patient class distribution:
label
NC    69
AD    33
Name: count, dtype: int64
Image class distribution:
label
NC    2440
AD    2000
Name: count, dtype: int64


In [19]:
# Recalculate patient IDs from the NEW splits
train_ids = set(train_df["patient_id"])
val_ids = set(val_df["patient_id"])
test_ids = set(test_df["patient_id"])

print("Train / Validation overlap:", len(train_ids & val_ids))
print("Train / Test overlap:", len(train_ids & test_ids))
print("Validation / Test overlap:", len(val_ids & test_ids))

assert not (train_ids & val_ids)
assert not (train_ids & test_ids)
assert not (val_ids & test_ids)

# Recalculate image paths from the NEW splits
train_paths = set(train_df["path"])
val_paths = set(val_df["path"])
test_paths = set(test_df["path"])

assert train_df["path"].is_unique
assert val_df["path"].is_unique
assert test_df["path"].is_unique

assert not (train_paths & val_paths)
assert not (train_paths & test_paths)
assert not (val_paths & test_paths)

# Verify that all candidate patients and images are retained
assert (
    train_ids | val_ids | test_ids
) == set(candidate_df["patient_id"])

assert (
    train_paths | val_paths | test_paths
) == set(candidate_df["path"])

assert (
    len(train_df) + len(val_df) + len(test_df)
) == len(candidate_df)

print("PASS: Patient-independent splits; all candidates retained.")

Train / Validation overlap: 0
Train / Test overlap: 0
Validation / Test overlap: 0
PASS: Patient-independent splits; all candidates retained.


# Step 6: Save Final Data Splits

Save the patient-independent training, validation, and test splits
as CSV files.

In [20]:
# Set output folder
output_folder = "/content/drive/MyDrive/COMP3710/project/dataset/splits"

os.makedirs(output_folder, exist_ok=True)

# Set output paths
train_path = os.path.join(output_folder, "train.csv")
val_path = os.path.join(output_folder, "val.csv")
test_path = os.path.join(output_folder, "test.csv")

# Update split labels
train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

# Save datasets
train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

# Display saved files
print("Training dataset saved to:", train_path)
print("Validation dataset saved to:", val_path)
print("Test dataset saved to:", test_path)

# Verify saved datasets
saved_train = pd.read_csv(train_path)
saved_val = pd.read_csv(val_path)
saved_test = pd.read_csv(test_path)

print("\nSaved dataset sizes:")
print("Training images:", len(saved_train))
print("Validation images:", len(saved_val))
print("Test images:", len(saved_test))

Training dataset saved to: /content/drive/MyDrive/COMP3710/project/dataset/splits/train.csv
Validation dataset saved to: /content/drive/MyDrive/COMP3710/project/dataset/splits/val.csv
Test dataset saved to: /content/drive/MyDrive/COMP3710/project/dataset/splits/test.csv

Saved dataset sizes:
Training images: 21620
Validation images: 4460
Test images: 4440


In [21]:
# Reload saved CSV files
saved_train = pd.read_csv(
    train_path,
    dtype={"image_id": str, "patient_id": str}
)

saved_val = pd.read_csv(
    val_path,
    dtype={"image_id": str, "patient_id": str}
)

saved_test = pd.read_csv(
    test_path,
    dtype={"image_id": str, "patient_id": str}
)

# Verify dataset sizes
print("Training images:", len(saved_train))
print("Validation images:", len(saved_val))
print("Test images:", len(saved_test))

Training images: 21620
Validation images: 4460
Test images: 4440


In [22]:
# Check class distribution
print("Training:")
print(train_df["label"].value_counts())
print(train_df["label"].value_counts(normalize=True) * 100)

print("\nValidation:")
print(val_df["label"].value_counts())
print(val_df["label"].value_counts(normalize=True) * 100)

print("\nTest:")
print(test_df["label"].value_counts())
print(test_df["label"].value_counts(normalize=True) * 100)

Training:
label
NC    11000
AD    10620
Name: count, dtype: int64
label
NC    50.878816
AD    49.121184
Name: proportion, dtype: float64

Validation:
label
AD    2240
NC    2220
Name: count, dtype: int64
label
AD    50.224215
NC    49.775785
Name: proportion, dtype: float64

Test:
label
NC    2440
AD    2000
Name: count, dtype: int64
label
NC    54.954955
AD    45.045045
Name: proportion, dtype: float64
